## Calculation of Spectral SNR ($SNR(f)$)

The `get_spectral_snr` function computes the continuous frequency-dependent **Spectral Signal-to-Noise Ratio** ($SNR(f)$). This calculation models the signal power distribution across the spectrum relative to the noise spectral density before any sampling or equalization takes place at the receiver.

### Expression

The function implements **Equation (10)** from the reference paper, given by:

$$SNR(f) = \frac{\frac{5}{36} \cdot T \cdot OMA_{outer}^2 \cdot |H_t(f)|^2 \cdot |H_{ch}(f)|^2}{S_N(f)}$$

Where:
* $T$ is the symbol period ($T = \frac{1}{R_s}$), where $R_s$ is the `symbol_rate`.
* $OMA_{outer}$ is the Outer Optical Modulation Amplitude (calculated from Equation 7).
* $H_t(f)$ is the frequency response of the transmitter.
* $H_{ch}(f)$ is the frequency response of the optical/electrical channel.
* $S_N(f)$ is the Power Spectral Density (PSD) of the noise.
* $\frac{5}{36}$ is the geometric scaling factor specific to the PAM-4 signaling mapping used in this IMDD framework.

---

In [1]:
import numpy as np

from oma_calculation import calculate_oma_outer

def get_spectral_snr(f, oma_outer, symbol_rate, h_t_f, h_ch_f, s_n_f):
    """
    Calcula o SNR espectral baseado na Equação (10)

    parâmetros:
    f: Frequência [Hz]
    oma_outer: OMA externa [mW]
    symbol_rate: Taxa de símbolos [Hz]
    h_t_f: Resposta em frequência do transmissor [adimensional]
    h_ch_f: Resposta em frequência do canal [adimensional]
    s_n_f: Densidade espectral de potência do ruído [W^2/Hz]
    retorna:
    snr_f: SNR espectral [adimensional]
    """
    t = 1 / symbol_rate  # Período do símbolo [cite: 89]
    
    # Numerador: 
    numerator = (5/36) * t * (oma_outer**2) * (np.abs(h_t_f)**2) * (np.abs(h_ch_f)**2)
    
    # SNR(f) = Numerador / S_N(f) [cite: 125]
    snr_f = numerator / s_n_f
    return snr_f

In [3]:
import numpy as np

# --- Execução do Teste de Validação para get_spectral_snr ---

print("--- Starting Validation Test for get_spectral_snr ---")

# 1. Definir parâmetros controlados de teste
f_test = np.array([-20, -10, 0, 10, 20])  # Vetor de frequências curto para o teste
oma_outer_test = 2.0                      # OMA externa = 2 mW
symbol_rate_test = 10.0                   # Rs = 10 Hz (T = 0.1 s)

# Respostas em frequência unitárias (planas) com o mesmo tamanho do vetor 'f'
h_t_test = np.ones_like(f_test)
h_ch_test = np.ones_like(f_test)
s_n_test = np.ones_like(f_test)           # Densidade de ruído unitária

# 2. Executar a função
calculated_snr = get_spectral_snr(
    f=f_test, 
    oma_outer=oma_outer_test, 
    symbol_rate=symbol_rate_test, 
    h_t_f=h_t_test, 
    h_ch_f=h_ch_test, 
    s_n_f=s_n_test
)

# 3. Valor teórico esperado para cada ponto do vetor: (5/36) * 0.1 * 4 / 1 = 1/18
expected_snr_val = 1.0 / 18.0
expected_snr_array = np.ones_like(f_test) * expected_snr_val

# 4. Exibir resultados na tela
print(f"Frequency vector: {f_test}")
print("-" * 50)
print(f"Calculated SNR (Linear): {calculated_snr}")
print(f"Expected SNR (Linear):   {expected_snr_array}")

# 5. Verificação automática do array usando o NumPy
if np.allclose(calculated_snr, expected_snr_array, rtol=1e-5):
    print("\n SUCCESS! The Spectral SNR algebra and array operations are correct.")
else:
    print("\n ERROR! The calculated array diverges from the theoretical value. Check the operations.")

--- Starting Validation Test for get_spectral_snr ---
Frequency vector: [-20 -10   0  10  20]
--------------------------------------------------
Calculated SNR (Linear): [0.05555556 0.05555556 0.05555556 0.05555556 0.05555556]
Expected SNR (Linear):   [0.05555556 0.05555556 0.05555556 0.05555556 0.05555556]

 SUCCESS! The Spectral SNR algebra and array operations are correct.
